```
CONFIGURATION MINIMALE RECOMMANDEE
Python  3.13.x ou 3.12.x
PySpark 4.x
pandas  2.2.3
Java    21.x ou 25.x
```


In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pathlib import Path
import os
from dotenv import load_dotenv

In [2]:
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 40g pyspark-shell"
load_dotenv()

True

In [ ]:
JDBC_DRIVER_PATH = (
    r"C:\Users\stgadmin\Documents\GitHub\Data-Engineering-project"
    r"\Drivers\sqljdbc_13.6.0.0_fra\sqljdbc_13.6\fra\jars"
    r"\mssql-jdbc-13.6.0.jre11.jar"
)

SQLSERVER_NATIVE_PATH = (
    r"C:\Users\stgadmin\Documents\GitHub\Data-Engineering-project"
    r"\Drivers\sqljdbc_13.6.0.0_fra\sqljdbc_13.6\auth\x64"
)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DomesticFlightDelays")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.jars", JDBC_DRIVER_PATH)
    # SQL Server JDBC
    .config("spark.driver.extraJavaOptions", f"-Djava.library.path={SQLSERVER_NATIVE_PATH}")
    .getOrCreate()
)


In [4]:
project_path = Path.cwd()
flight_folder_path = project_path / "Data" / "Bronze" / "Stats_US_Flights_2019_2026"
passenger_folder_path = project_path / "Data" / "Bronze"/ "T_T100D_MARKET_US_CARRIER_ONLY"
runways_folder_path = project_path / "Data" / "Bronze" / "Runways"

flight_silver_path = project_path / "Data" / "Silver" / "Flights"
passengers_silver_path = project_path / "Data" / "Silver" / "Passengers"

In [5]:
flight_files = [
    file.as_posix()
    for file in flight_folder_path.glob("*.csv")
]

passenger_files = [
    file.as_posix()
    for file in passenger_folder_path.glob("*.csv")
]

runways_files = [
    file.as_posix()
    for file in runways_folder_path.glob("*.csv")
]


In [6]:
COLUMNS_TO_KEEP_FLIGHTS = [
    "Year",
    "Quarter",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Reporting_Airline",
    "DOT_ID_Reporting_Airline",
    "OriginAirportID",
    "Origin",
    "OriginCityName",
    "OriginState",
    "OriginStateName",
    "DestAirportID",
    "Dest",
    "DestCityName",
    "DestState",
    "DestStateName",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "Cancelled",
    "CancellationCode",
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance",
    "DistanceGroup",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "DivAirportLandings",
    "DivReachedDest",
    "DivActualElapsedTime",
    "DivArrDelay",
    "DivDistance"
]


FLIGHT_SCHEMA = StructType([

    StructField("Year", IntegerType(), True),
    StructField("Quarter", IntegerType(), True),
    StructField("Month", IntegerType(), True),
    StructField("DayofMonth", IntegerType(), True),
    StructField("DayOfWeek", IntegerType(), True),
    StructField("FlightDate", DateType(), True),

    StructField("Reporting_Airline", StringType(), True),
    StructField("DOT_ID_Reporting_Airline", IntegerType(), True),

    StructField("OriginAirportID", IntegerType(), True),
    StructField("Origin", StringType(), True),
    StructField("OriginCityName", StringType(), True),
    StructField("OriginState", StringType(), True),
    StructField("OriginStateName", StringType(), True),

    StructField("DestAirportID", IntegerType(), True),
    StructField("Dest", StringType(), True),
    StructField("DestCityName", StringType(), True),
    StructField("DestState", StringType(), True),
    StructField("DestStateName", StringType(), True),

    StructField("CRSDepTime", IntegerType(), True),
    StructField("DepTime", IntegerType(), True),
    StructField("DepDelay", DoubleType(), True),

    StructField("TaxiOut", DoubleType(), True),
    StructField("WheelsOff", IntegerType(), True),
    StructField("WheelsOn", IntegerType(), True),
    StructField("TaxiIn", DoubleType(), True),

    StructField("CRSArrTime", IntegerType(), True),
    StructField("ArrTime", IntegerType(), True),
    StructField("ArrDelay", DoubleType(), True),

    StructField("Cancelled", DoubleType(), True),
    StructField("CancellationCode", StringType(), True),

    StructField("CRSElapsedTime", DoubleType(), True),
    StructField("ActualElapsedTime", DoubleType(), True),
    StructField("AirTime", DoubleType(), True),

    StructField("Distance", DoubleType(), True),
    StructField("DistanceGroup", DoubleType(), True),

    StructField("CarrierDelay", DoubleType(), True),
    StructField("WeatherDelay", DoubleType(), True),
    StructField("NASDelay", DoubleType(), True),
    StructField("SecurityDelay", DoubleType(), True),
    StructField("LateAircraftDelay", DoubleType(), True),

    StructField("DivAirportLandings", IntegerType(), True),
    StructField("DivReachedDest", DoubleType(), True),
    StructField("DivActualElapsedTime", DoubleType(), True),
    StructField("DivArrDelay", DoubleType(), True),
    StructField("DivDistance", DoubleType(), True)
])

In [7]:
FLIGHT_COLUMN_MAPPING = {
    "Year": "YEAR",
    "Quarter": "QUARTER",
    "Month": "MONTH",
    "DayofMonth": "DAY_OF_MONTH",
    "DayOfWeek": "DAY_OF_WEEK",
    "FlightDate": "FLIGHT_DATE",

    "Reporting_Airline": "REPORTING_AIRLINE",
    "DOT_ID_Reporting_Airline": "DOT_ID_REPORTING_AIRLINE",

    "OriginAirportID": "ORIGIN_AIRPORT_ID",
    "Origin": "ORIGIN",
    "OriginCityName": "ORIGIN_CITY_NAME",
    "OriginState": "ORIGIN_STATE",
    "OriginStateName": "ORIGIN_STATE_NAME",

    "DestAirportID": "DEST_AIRPORT_ID",
    "Dest": "DEST",
    "DestCityName": "DEST_CITY_NAME",
    "DestState": "DEST_STATE",
    "DestStateName": "DEST_STATE_NAME",

    "CRSDepTime": "CRS_DEP_TIME",
    "DepTime": "DEP_TIME",
    "DepDelay": "DEP_DELAY",

    "TaxiOut": "TAXI_OUT",
    "WheelsOff": "WHEEL_OFF",
    "WheelsOn": "WHEEL_ON",
    "TaxiIn": "TAXI_IN",

    "CRSArrTime": "CRS_ARR_TIME",
    "ArrTime": "ARR_TIME",
    "ArrDelay": "ARR_DELAY",

    "Cancelled": "CANCELLED",
    "CancellationCode": "CANCELLATION_CODE",

    "CRSElapsedTime": "CRS_ELAPSED_TIME",
    "ActualElapsedTime": "ACTUAL_ELAPSED_TIME",
    "AirTime": "AIR_TIME",

    "Distance": "DISTANCE",
    "DistanceGroup": "DISTANCE_GROUP",

    "CarrierDelay": "CARRIER_DELAY",
    "WeatherDelay": "WEATHER_DELAY",
    "NASDelay": "NAS_DELAY",
    "SecurityDelay": "SECURITY_DELAY",
    "LateAircraftDelay": "LATE_AIRCRAFT_DELAY",

    "DivAirportLandings": "DIV_AIRPORT_LANDINGS",
    "DivReachedDest": "DIV_REACHED_DEST",
    "DivActualElapsedTime": "DIV_ACTUAL_ELAPSED_TIME",
    "DivArrDelay": "DIV_ARR_DELAY",
    "DivDistance": "DIV_DISTANCE"
}

In [8]:
df_flights = (spark.read.csv(
        flight_files,
        header=True,
        inferSchema=False
    )
    .select(COLUMNS_TO_KEEP_FLIGHTS)
)

In [9]:
df_flights = df_flights.select(* [F.col(field.name).cast(field.dataType).alias(FLIGHT_COLUMN_MAPPING[field.name])for field in FLIGHT_SCHEMA.fields])

In [10]:
COLUMNS_TO_KEEP_PASSENGERS = [
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED",
    "PAYLOAD",
    "SEATS",
    "PASSENGERS",
    "FREIGHT",
    "MAIL",
    "DISTANCE",
    "AIR_TIME",
    "UNIQUE_CARRIER",
    "AIRLINE_ID",
    "CARRIER_NAME",
    "ORIGIN_AIRPORT_ID",
    "ORIGIN",
    "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_ABR",
    "ORIGIN_STATE_NM",
    "DEST_AIRPORT_ID",
    "DEST",
    "DEST_CITY_NAME",
    "DEST_STATE_ABR",
    "DEST_STATE_NM",
    "YEAR",
    "QUARTER",
    "MONTH",
    "DISTANCE_GROUP"
]

PASSENGERS_SCHEMA = StructType([
    StructField("DEPARTURES_SCHEDULED", DoubleType(), True),
    StructField("DEPARTURES_PERFORMED", DoubleType(), True),
    StructField("PAYLOAD", DoubleType(), True),
    StructField("SEATS", DoubleType(), True),
    StructField("PASSENGERS", DoubleType(), True),
    StructField("FREIGHT", DoubleType(), True),
    StructField("MAIL", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("UNIQUE_CARRIER", StringType(), True),
    StructField("AIRLINE_ID", IntegerType(), True),
    StructField("CARRIER_NAME", StringType(), True),
    StructField("ORIGIN_AIRPORT_ID", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("ORIGIN_CITY_NAME", StringType(), True),
    StructField("ORIGIN_STATE_ABR", StringType(), True),
    StructField("ORIGIN_STATE_NM", StringType(), True),
    StructField("DEST_AIRPORT_ID", IntegerType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY_NAME", StringType(), True),
    StructField("DEST_STATE_ABR", StringType(), True),
    StructField("DEST_STATE_NM", StringType(), True),
    StructField("YEAR", IntegerType(), True),
    StructField("QUARTER", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DISTANCE_GROUP", IntegerType(), True)
])

In [11]:
df_passengers = (spark.read.csv(
        passenger_files,
        header=True,
        inferSchema=False,
    ).select(COLUMNS_TO_KEEP_PASSENGERS)
)

In [12]:
df_passengers = df_passengers.select(*[F.col(field.name)
                                       .cast(field.dataType)
                                       .alias(field.name) for field in PASSENGERS_SCHEMA.fields])


In [13]:
COLUMNS_TO_KEEP_AIRPORTS = [
    "OID",
    "STATE_CODE",
    "ARPT_ID",
    "ARPT_NAME",
    "CITY",
    "COUNTRY_CODE",
    "LAT1_DECIMAL",
    "LONG1_DECIMAL"
]

In [14]:
df_runways = spark.read.csv(runways_files, inferSchema= True, header= True).select(COLUMNS_TO_KEEP_AIRPORTS)

In [15]:
df_runways = df_runways.drop_duplicates(["ARPT_ID"])
rename_map = {
    "LAT1_DECIMAL": "LATITUDE",
    "LONG1_DECIMAL": "LONGITUDE"
}
df_runways = df_runways.withColumnsRenamed(rename_map)

In [16]:
df_flights = df_flights.drop_duplicates()

In [17]:
first_condition = F.col("DEP_DELAY").isNull() | F.col("ARR_DELAY").isNull()
second_condition = F.col("CRS_ELAPSED_TIME").isNull() | F.col("ACTUAL_ELAPSED_TIME").isNull()
third_condition = F.col("CANCELLED") == 1.00

df_flights = df_flights.filter(third_condition | ~(first_condition & second_condition))

In [18]:
df_flights = df_flights.join(df_runways, (df_flights["ORIGIN"] == df_runways["ARPT_ID"]) & (df_flights["ORIGIN_STATE"] == df_runways["STATE_CODE"]),
                              "inner").select(df_flights["*"], df_runways["LATITUDE"], df_runways["LONGITUDE"])

In [19]:
df_passengers = df_passengers.filter(F.col("PASSENGERS") != 0.0).sort("PASSENGERS")
df_passengers = df_passengers.drop_duplicates()

In [20]:
df_passengers = df_passengers.repartition("YEAR", "MONTH")

(df_passengers.write
    .mode("overwrite")
    .partitionBy("YEAR", "MONTH")
    .parquet(passengers_silver_path.as_posix())
)

Py4JJavaError: An error occurred while calling o804.parquet.
: org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to file:/c:/Users/stgadmin/Documents/GitHub/Data-Engineering-project/Data/Silver/Passengers. SQLSTATE: 58030
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:843)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.enrichWriteError(FileFormatDataWriter.scala:92)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.commit(FileFormatDataWriter.scala:123)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeTask$1(FileFormatWriter.scala:416)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:1337)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:427)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$

In [ ]:
df_flights = df_flights.repartition("YEAR", "MONTH")

(df_flights.write
    .mode("overwrite")
    .partitionBy("YEAR", "MONTH")
    .parquet(flight_silver_path.as_posix())
)

In [ ]:
df_flights_silver = spark.read.parquet(flight_silver_path.as_posix())
df_passengers_silver =  spark.read.parquet(passengers_silver_path.as_posix())

Py4JJavaError: An error occurred while calling o64.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2079)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2123)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:218)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:132)
	at scala.collection.immutable.List.map(L

In [ ]:
first_columns = ["YEAR", "MONTH", "QUARTER"]

other_columns_passengers = [
    col for col in df_passengers_silver.columns
    if col not in first_columns
]

other_columns_flights = [
    col for col in df_flights_silver.columns
    if col not in first_columns
]

df_passengers_silver = df_passengers_silver.select(*first_columns, *other_columns_passengers)

df_flights_silver = df_flights_silver.select(*first_columns, *other_columns_flights)


In [15]:
sql_user = os.getenv("SQLSERVER_USER")
sql_password = os.getenv("SQLSERVER_PASSWORD")
sql_database = os.getenv("SQLSERVER_DATABASE")
sql_table = os.getenv("SQL_SERVER_TABLE")

sql_server_url = (
    "jdbc:sqlserver://ICT-210-05;"
    "databaseName=StagingSilver;"
    "integratedSecurity=true;"
    "encrypt=false;"
    "trustServerCertificate=true"
)


(
    df_flights_silver.write
        .format("jdbc")
        .option("url", sql_server_url)
        .option("dbtable", sql_table)
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .option("batchsize", "10000")
        .option("numPartitions", "4")
        .mode("append")
        .save()
)


Py4JJavaError: An error occurred while calling o175.save.
: com.microsoft.sqlserver.jdbc.SQLServerException: Ce pilote n'est pas configuré pour l'authentification intégrée. ClientConnectionId:b83c1d55-dfa5-4d35-b18a-6e52bec566f2
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.terminate(SQLServerConnection.java:5014)
	at com.microsoft.sqlserver.jdbc.AuthenticationJNI.<init>(AuthenticationJNI.java:78)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.logon(SQLServerConnection.java:6334)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection$LogonCommand.doExecute(SQLServerConnection.java:6322)
	at com.microsoft.sqlserver.jdbc.TDSCommand.execute(IOBuffer.java:8005)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.executeCommand(SQLServerConnection.java:5138)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.connectHelper(SQLServerConnection.java:4522)
	at com.microsoft.sqlserver.jdbc.SQLServerConnection.login(SQLServerConnection.java:4045)
	at com.microsoft.sqlserver